#ETL EM UM E-COMMERCE
Você foi contratado como Analista de Dados Júnior em uma empresa de e-commerce.

A empresa exportou um relatório chamado:
Vendas_Teste_ETL.xlsx

Mas o problema é:

⚠ O arquivo está cheio de problemas:

*   Linhas em branco
*   Datas inválidas
*   Valores com "R$"
*   Cidades com letras maiúsculas/minúsculas misturadas
*   Registros duplicados
*   Valores negativos

Seu desafio é:
Limpar os dados para que a empresa consiga gerar relatórios confiáveis.

##Parte 1: Importar dados e bibliotecas


In [1]:
import pandas as pd
import numpy as np

arquivo = 'Vendas_Teste_ETL.xlsx'

df = pd.read_excel(arquivo, sheet_name='Vendas')
print(df.columns)
print(df.shape)
df.head()

Index(['data_venda', 'pedido_id', 'cliente_id', 'produto_id', 'categoria',
       'vendedor_id', 'cidade', 'estado', 'canal_venda', 'forma_pagamento',
       'quantidade', 'valor_unitario', 'desconto_pct', 'valor_bruto',
       'valor_liquido', 'custo_unitario', 'margem', 'status_pedido',
       'data_pagamento', 'data_entrega_prevista', 'data_entrega_real',
       'dias_entrega', 'atraso_entrega'],
      dtype='object')
(3035, 23)


,data_venda,pedido_id,cliente_id,produto_id,categoria,vendedor_id,cidade,estado,canal_venda,forma_pagamento,...,valor_bruto,valor_liquido,custo_unitario,margem,status_pedido,data_pagamento,data_entrega_prevista,data_entrega_real,dias_entrega,atraso_entrega
0,2025-10-15 00:00:00,2034050.0,1738.0,202.0,Livros,285.0,Recife,PE,Marketplace,Crédito,...,1499.0,1364.09,713.89,650.20,Devolvida,2025-10-15 00:00:00,2025-10-18,2025-10-22,7.0,1.0
1,2025-08-03 00:00:00,1968504.0,1333.0,403.0,Brinquedos,669.0,Rio de Janeiro,RJ,Loja Física,Pix,...,3599.0,2987.17,2359.86,627.31,Cancelada,2025-08-04 00:00:00,2025-08-11,2025-08-11,7.0,0.0
2,2025-01-03 00:00:00,1843895.0,1835.0,109.0,Esportes,265.0,Curitiba,PR,E-commerce,Débito,...,1998.0,1818.18,771.29,275.60,Cancelada,2025-01-03 00:00:00,2025-01-13,2025-01-12,9.0,0.0
3,2025-05-23 00:00:00,1913377.0,1145.0,301.0,Brinquedos,215.0,FORTALEZA,CE,Televendas,Pix,...,1199.0,971.19,763.74,207.45,Concluída,2025-05-23 00:00:00,2025-06-02,2025-06-04,12.0,1.0
4,2025-09-23 00:00:00,2094564.0,1006.0,101.0,Moda,144.0,São Paulo,SP,Loja Física,Boleto,...,699.0,573.18,315.61,257.57,Devolvida,2025-09-26 00:00:00,2025-10-02,2025-10-05,9.0,1.0


##Parte 2: Limpar dados


###2.1: Remover linhas vazias

In [2]:
print(df.isna().sum())
df = df.dropna(how='all')
print(df.isna().sum())


data_venda               10
pedido_id                10
cliente_id               10
produto_id               10
categoria                50
vendedor_id              10
cidade                   51
estado                   10
canal_venda              10
forma_pagamento          50
quantidade               50
valor_unitario           63
desconto_pct             10
valor_bruto              10
valor_liquido            50
custo_unitario           10
margem                   10
status_pedido            10
data_pagamento           10
data_entrega_prevista    10
data_entrega_real        10
dias_entrega             10
atraso_entrega           10
dtype: int64
data_venda                0
pedido_id                 0
cliente_id                0
produto_id                0
categoria                40
vendedor_id               0
cidade                   41
estado                    0
canal_venda               0
forma_pagamento          40
quantidade               40
valor_unitario           53
descont

###2.2: Padronizar cidades

In [3]:
print(df['cidade'].unique())
df.loc[:, 'cidade'] = df['cidade'].astype(str)
df.loc[:, 'cidade'] = df['cidade'].str.strip()
df.loc[:, 'cidade'] = df['cidade'].str.lower().str.title()
print(df['cidade'].unique())

['Recife' 'Rio de Janeiro' 'Curitiba' 'FORTALEZA' 'São Paulo' 'SALVADOR'
 'Fortaleza' 'caldas novas' 'Brasília' 'Anápolis' 'Belo Horizonte' nan
 'Goiânia' 'Salvador' 'Uberlândia' 'RECIFE' 'Caldas Novas' '  Goiânia  '
 '  Belo Horizonte  ' 'brasília' 'GOIÂNIA' 'CALDAS NOVAS' 'recife'
 'BELO HORIZONTE' '  Recife  ' '  Salvador  ' 'uberlândia' 'ANÁPOLIS'
 'SÃO PAULO' '  Uberlândia  ' 'RIO DE JANEIRO' '  Caldas Novas  '
 'CURITIBA' 'fortaleza' 'rio de janeiro' '  Fortaleza  ' '  São Paulo  '
 'UBERLÂNDIA' 'anápolis' 'são paulo' 'belo horizonte' 'salvador'
 '  Rio de Janeiro  ' 'BRASÍLIA' 'goiânia' '  Anápolis  ' '  Curitiba  '
 '  Brasília  ' 'curitiba']
['Recife' 'Rio De Janeiro' 'Curitiba' 'Fortaleza' 'São Paulo' 'Salvador'
 'Caldas Novas' 'Brasília' 'Anápolis' 'Belo Horizonte' 'Nan' 'Goiânia'
 'Uberlândia']


###2.3: Padronizar valores unitários

In [4]:
print(df['valor_unitario'].unique())
df.loc[:, 'valor_unitario'] = df['valor_unitario'].astype(str)
df.loc[:, 'valor_unitario'] = (df['valor_unitario']
                               .str.replace('R$','',regex=False)
                               .str.replace('.','',regex=False)
                               .str.replace(',','.',regex=False)
)
df.loc[:, 'valor_unitario'] = pd.to_numeric(df['valor_unitario'], errors='coerce')
df['valor_unitario'].head()

[1499 3599 999 1199 699 '1499,00' 1299 799 589 nan 749 99999 189 469 1799
 219 2899 279 'R$ 219.00' '999,00' 'R$ 999.00' 'R$ 1499.00' '589,00'
 '3599,00' '1199,00' 'R$ 589.00' '699,00' '799,00' '279,00' '1799,00'
 '2899,00' 'R$ 189.00' 'R$ 279.00' 'R$ 699.00' 'R$ 749.00' '749,00'
 'R$ 799.00' 'R$ 1199.00' 'R$ 1799.00' '1299,00' 'R$ 1299.00' 'R$ 469.00'
 '469,00' '189,00' 'R$ 3599.00']


,valor_unitario
0,1499.0
1,3599.0
2,999.0
3,1199.0
4,699.0


###2.4: Remover datas inválidas

In [5]:
print(df['data_venda'].head(10))
df.loc[:, 'data_venda'] = pd.to_datetime(df['data_venda'],errors='coerce',dayfirst=True)
print('Datas inválidas:',df['data_venda'].isna().sum())

0    2025-10-15 00:00:00
1    2025-08-03 00:00:00
2    2025-01-03 00:00:00
3    2025-05-23 00:00:00
4    2025-09-23 00:00:00
5    2025-08-25 00:00:00
6    2025-01-27 00:00:00
7    2025-04-02 00:00:00
8    2025-06-21 00:00:00
9    2025-04-21 00:00:00
Name: data_venda, dtype: object
Datas inválidas: 15


###2.5: Remover dados duplicados

In [6]:
print(df.duplicated().sum())
df = df.drop_duplicates()
print('Depois de ter sido limpo:',df.duplicated().sum())

25
Depois de ter sido limpo: 0


###2.6: Remover valores negativos

In [7]:
print('Valores de quantidade negativa antes da correção:')
display(df[df['quantidade'] < 0]['quantidade'])

df.loc[df["quantidade"] < 0, "quantidade"] = 0

print('\nValores de quantidade negativa depois da correção (deve estar vazio):')
display(df[df['quantidade'] < 0]['quantidade'])

Valores de quantidade negativa antes da correção:


,quantidade
181,-1.0
429,-1.0
675,-1.0
2360,-1.0
2770,-1.0



Valores de quantidade negativa depois da correção (deve estar vazio):


,quantidade


##3: Salvar os dados limpos

In [9]:
print("\nFormato final:", df.head)
print(df.describe)


Formato final: <bound method NDFrame.head of                data_venda  pedido_id  cliente_id  produto_id       categoria  \
0     2025-10-15 00:00:00  2034050.0      1738.0       202.0          Livros   
1     2025-08-03 00:00:00  1968504.0      1333.0       403.0      Brinquedos   
2     2025-01-03 00:00:00  1843895.0      1835.0       109.0        Esportes   
3     2025-05-23 00:00:00  1913377.0      1145.0       301.0      Brinquedos   
4     2025-09-23 00:00:00  2094564.0      1006.0       101.0            Moda   
...                   ...        ...         ...         ...             ...   
3005  2025-07-02 00:00:00  1939993.0      1900.0       105.0      Brinquedos   
3006  2025-12-25 00:00:00  2104699.0      1506.0       108.0  Casa e Cozinha   
3007  2025-07-12 00:00:00  2004306.0      1739.0       106.0          Beleza   
3008  2025-11-17 00:00:00  2080677.0      1389.0       203.0          Beleza   
3009  2025-10-21 00:00:00  2120854.0      1383.0       202.0      Automoti